# Redis的开发

## 驱动与连接

In [1]:
import redis

In [2]:
r = redis.Redis(
    host="127.0.0.1",
    port=6379,
    db=0,  # 数据库名 1-15可供选择。
    decode_responses=True  # 是否自动解码（从二进制还原成字数串）视情况而定，如果是二进制数据，则False
)

In [3]:
r.ping()

True

### 连接池

In [5]:
pool = redis.ConnectionPool(
    host="127.0.0.1",
    port=6379,
    db=0,
    max_connections=10,
    decode_responses=True
)

In [6]:
r = redis.Redis(connection_pool=pool)

try:
    r.ping()
    print("success")
except:
    print("fail")

success


## 数据类型

| 数据类型 | 核心特性 | 典型应用场景 |
| :--- | :--- | :--- |
| **String (字符串)** | 最基础类型，二进制安全，可存储文本、数字、序列化对象等，最大支持 **512MB** 。 | 缓存、计数器、分布式锁、Session存储 。 |
| **Hash (哈希)** | 键值对集合，类似编程语言中的字典或Map，适合存储对象 。 | 存储和修改用户信息、商品属性、配置项 。 |
| **List (列表)** | 有序的字符串列表，按插入顺序排序，底层为双向链表 。 | 消息队列、最新动态列表、历史记录 。 |
| **Set (集合)** | 无序、元素唯一（自动去重）的字符串集合 。 | 标签系统、共同好友、抽奖去重、黑白名单 。 |
| **Sorted Set (有序集合)** | 在Set基础上，每个元素关联一个`score`（分数），并按分数排序 。 | 实时排行榜、带权重的任务调度 。 |
| **Bitmap (位图)** | 基于String的位操作，可对字符串的位（bit）进行设置和获取 。 | 用户签到、活跃用户统计、布隆过滤器实现 。 |
| **HyperLogLog** | 概率数据结构，用极小内存（固定约12KB）估算海量数据的基数（去重数量），有约**0.81%**的标准误差 。 | 统计网站UV（独立访客）、大规模去重计数 。 |
| **GEO (地理空间)** | 基于Sorted Set实现，用于存储地理位置并执行距离计算、半径查询等操作 。 | “附近的人”、门店搜索、地理围栏 。 |
| **Stream (流)** | 类似消息队列的只追加日志数据结构，支持消费者组 。 | 事件溯源、日志聚合、微服务间通信 。 |
| **JSON** | 提供对JSON数据结构的原生支持，可对文档内的单个元素进行增删改查 。 | 存储和操作复杂的、嵌套的文档型数据。 |
| **Time Series (时间序列)** | 专门用于存储和查询带时间戳的数据点 。 | 监控指标、IoT设备数据采集与分析。 |
| **Bloom/Cuckoo Filter (布隆/布谷鸟过滤器)** | 概率数据结构，用于判断一个元素是否**可能**存在于一个集合中，内存占用少 。 | 防止缓存穿透 。 |
| **Vector Set (矢量集合)** | 专门管理高维矢量数据，支持高效的相似性搜索（如HNSW算法）。 | 推荐系统、语义搜索、AI应用 。 |

- 重点
    - 增删改查
    - 操作

### 基本数据类型

- 主要函数
    - set/get
    - mset/mget
    - incr++/incrb+=
    - decr--/decrby-=

基本数据存储

In [7]:
r.set(name="age", value=18, ex=30) # ex是数据过期时间

True

In [10]:
r.get(name="age")

In [11]:
r.mset({
    "name": "wjc",
    "score": 100,
    "subject": "math"
})

True

In [12]:
r.mget(["name", "subject", "score"])

['wjc', 'math', '100']

In [13]:
r.decr("score")

99

In [14]:
r.incrby("score", 1)

100

对象存储

In [15]:
import redis
import pickle

class Person:
    def __init__(self, name, scores):
        self.name = name  # 字段必须可序列化，1.字节流 2.json，就是指数据的存储方式
        self.scores = scores
    
    def greet(self):
        return f"你好, {self.name}"

p = Person(name="wjc", scores=[100, 100])

In [17]:
r = redis.Redis(
    host="127.0.0.1",
    port=6379,
    db=0,  # 数据库名 1-15可供选择。
    decode_responses=False  # 是否自动解码（从二进制还原成字数串）视情况而定，如果是二进制数据，则False
)
r.set("person:01", pickle.dumps(p))

True

In [18]:
data = r.get("person:01")
p_restored = pickle.loads(data)
p_restored.greet()

'你好, wjc'

json数据存储

In [19]:
import redis
import json

# 1. 连接 Redis
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

# 2. 定义一个可序列化的对象（字典或类实例）
class User:
    def __init__(self, name, age, email):
        self.name = name
        self.age = age
        self.email = email
    
    def to_dict(self):
        return {"name": self.name, "age": self.age, "email": self.email}
    
    @classmethod
    def from_dict(cls, data):
        return cls(data["name"], data["age"], data["email"])

# 3. 存储对象
user = User("靓仔", 30, "langzai@qq.com")
r.set("user:1001", json.dumps(user.to_dict()))

# 4. 读取并还原对象
data = r.get("user:1001")
if data:
    user_restored = User.from_dict(json.loads(data))
    print(user_restored.name, user_restored.age)  # Alice 30

靓仔 30


### hash

In [4]:
r.hset("01", 'name', 'wjc')

1

In [10]:
r.hset("02", mapping={
    'name': 'wjc',
    'age': 18
})

0

In [12]:
r.hset("03", items=[('name', 'wjc'), ('age', 18)])

In [11]:
# 获取
print(r.hget('user:1', 'name'))  # 输出: 李四
print(r.hgetall('user:2'))  # 输出所有字段
print(r.hmget('user:1', ['name', 'age']))  # 输出: ['李四', '25']

2

In [ ]:
print(r.hkeys('user:1'))  # 输出所有key
print(r.hvals('user:1'))  # 输出所有value

### list

In [23]:
import redis
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

In [24]:
# 插入元素
r.lpush('mylist', 'first', 'second', 'third')  # 从左边插入
r.rpush('mylist', 'fourth', 'fifth')  # 从右边插入

5

In [25]:
# 获取元素
print(r.lrange('mylist', 0, -1))  # 获取所有
print(r.lindex('mylist', 0))  # 获取第一个元素

['third', 'second', 'first', 'fourth', 'fifth']
third


In [26]:
# 弹出元素
print(r.lpop('mylist'))  # 从左边弹出
print(r.rpop('mylist'))  # 从右边弹出

third
fifth


In [27]:
# 获取列表长度
print(r.llen('mylist'))

3


### Set

- 集合特点是元素唯一
    - 支持集合运算

In [28]:
# 添加元素
r.sadd('myset', 'apple', 'banana', 'orange', 'apple')  # 重复会自动去重
r.sadd('myset', 1, 3, 6, 'apple')  # 重复会自动去重

3

In [29]:
# 获取所有元素
print(r.smembers('myset'))  # 输出: {'banana', 'orange', 'apple'}

{'1', 'orange', '6', 'apple', '3', 'banana'}


In [30]:
# 检查元素是否存在
print(r.sismember('myset', 'apple'))  # 输出: True

1


In [31]:
# 集合运算
r.sadd('set1', 'a', 'b', 'c')
r.sadd('set2', 'b', 'c', 'd')

3

In [32]:
print(r.sinter('set1', 'set2'))  # 交集: {'b', 'c'}
print(r.sunion('set1', 'set2'))  # 并集: {'a', 'b', 'c', 'd'}
print(r.sdiff('set1', 'set2'))   # 差集: {'a'}

{'b', 'c'}
{'d', 'b', 'c', 'a'}
{'a'}


### sorted Set 

- 除了有序外，其中每个元素都具有字段`score`
    - 顺序由`score`决定

In [39]:
# 添加元素（带分数）
r.zadd('leaderboard', {'Alice': 100, 'Bob': 85, 'Charlie': 95, 'CJW': 95, 'CJA': 95})

1

In [41]:
# 获取排名
print(r.zrank('leaderboard', 'CJA'))  # 输出: 2 (从0开始)
# low -> high 同score比较每位字符大小排序

1


In [42]:
# 按分数范围获取
print(r.zrange('leaderboard', 0, -1, withscores=True))
# 输出: [('Bob', 85.0), ('Charlie', 95.0), ('Alice', 100.0)]

[('Bob', 85.0), ('CJA', 95.0), ('CJW', 95.0), ('Charlie', 95.0), ('Alice', 100.0)]


In [43]:
# 增加分数
r.zincrby('leaderboard', 10, 'Bob')  # Bob增加10分

95.0

### 位图

In [ ]:
# 1. SETBIT: 设置某个偏移位的值 (0 或 1)
# 将 key 为 'user:online' 的位图中，偏移量为 123 的位设为 1
r.setbit('user:online', 123, 1)   # 第二个参数必须是整数。（如果想放字符串，可以使用hash函数把字符串转换为非负整数）
print("获取偏移量的值：", r.getbit('user:online', 123))

In [ ]:
# 3. BITCOUNT: 统计位图中所有值为 1 的位的数量
online_count = r.bitcount('user:online')
print(f"当前在线用户总数: {online_count}")  # 输出: 1

使用场景
每日活跃用户

每日活跃

In [44]:
from datetime import datetime

def user_visited(user_id: int):
    """记录某个用户在今天访问了"""
    today = datetime.now().strftime('%Y-%m-%d')
    key = f"dau:{today}"
    # 假设用户ID直接作为偏移量 (确保ID是整数且非负)
    r.setbit(key, user_id, 1)  # 1可以是其他数，实际值区分0与非0

def get_dau_count(date_str: str) -> int:
    """获取某天的日活用户数"""
    key = f"dau:{date_str}"
    return r.bitcount(key)

# 模拟用户访问
user_visited(1001)
user_visited(1002)
user_visited(1005)

today = datetime.now().strftime('%Y-%m-%d')
print(f"今日活跃用户数: {get_dau_count(today)}")  # 输出: 3

今日活跃用户数: 3


连续活跃

In [45]:
# 模拟两天DAU数据
# 第一天：用户 1, 2, 3 活跃
r.setbit('dau:2026-09-03', 1, 1)
r.setbit('dau:2026-09-03', 2, 1)
r.setbit('dau:2026-09-03', 3, 1)
# 第二天：用户 2, 3, 4 活跃
r.setbit('dau:2026-09-04', 2, 1)
r.setbit('dau:2026-09-04', 3, 1)
r.setbit('dau:2026-09-04', 4, 1)

# 用 BITOP 计算连续两天都活跃的用户 (AND运算)
# 结果存放在临时键 'dau:retained:2026-09-04' 中
r.bitop('AND', 'dau:retained:2026-09-04', 'dau:2026-09-03', 'dau:2026-09-04') # 多个日期统计
# r.bitop('AND', 'dau:retained:2026-09-04', *['dau:2026-09-03', 'dau:2026-09-04', "dau:2026-09-05"])   # 更多的运算，参考： https://redis.io/commands/bitop

retained_count = r.bitcount('dau:retained:2026-09-04')
print(f"两天连续活跃用户数: {retained_count}")  # 输出: 2 (用户2和3)

两天连续活跃用户数: 2


### HyperLogLog

用很少的资源，统计不重复的元素

In [ ]:
# 1. PFADD: 添加元素（可以是单个或多个）
r.pfadd('uv:2026-09-04', 'user_123', 'user_456', 'user_789')

# 2. PFCOUNT: 获取去重后的基数（估算值）
count = r.pfcount('uv:2026-09-04')
print(count)  # 输出: 3

# 3. PFMERGE: 合并多个 HyperLogLog（用于多维度或时间范围统计）
r.pfmerge('uv:week1', 'uv:2026-09-01', 'uv:2026-09-02', 'uv:2026-09-03')

记录用户访问

In [ ]:
from datetime import datetime, timedelta
def track_user_visit(user_id: str):
    """记录用户访问（独立访客统计）"""
    today = datetime.now().strftime('%Y-%m-%d')
    key = f"uv:{today}"
    # PFADD 会自动去重
    r.pfadd(key, user_id)
    print(f"用户 {user_id} 已记录")

# 模拟用户访问
track_user_visit('user_001')
track_user_visit('user_002')
track_user_visit('user_001')  # 重复访问，不会增加计数

记录访客

In [ ]:
def get_today_uv() -> int:
    today = datetime.now().strftime('%Y-%m-%d')
    return r.pfcount(f"uv:{today}")

uv_today = get_today_uv()
print(f"今日独立访客 (UV): 约 {uv_today} 人")

n天的访问客户

In [ ]:
def get_last_7_days_uv() -> int:
    """统计最近7天的独立访客总数 (去重)"""
    today = datetime.now()
    week_keys = []
    
    for i in range(7):
        day = (today - timedelta(days=i)).strftime('%Y-%m-%d')
        week_keys.append(f"uv:{day}")
    
    # 将 7 天的 HLL 合并到临时键 'uv:last_7_days' 中
    r.pfmerge('uv:last_7_days', *week_keys)
    
    # 统计合并后的基数
    total_uv = r.pfcount('uv:last_7_days')
    
    # 清理临时键 (可选)
    # r.delete('uv:last_7_days')
    return total_uv

print(f"近7天独立访客 (UV): 约 {get_last_7_days_uv()} 人")

### GEO(地理空间)

In [46]:
# 1. GEOADD: 添加地理位置 (经度, 纬度, 成员名)
r.geoadd('locations', [116.397, 39.908, 'Beijing'])
r.geoadd('locations', [121.473, 31.230, 'Shanghai'])

1

In [47]:
# 2. GEODIST: 计算两点间距离 (单位: m/km/mi/ft)
distance = r.geodist('locations', 'Beijing', 'Shanghai', unit='km')
print(f"北京到上海距离: {distance} km")

北京到上海距离: 1068.4038 km


In [48]:
# 3. GEOPOS: 获取某个成员的经纬度
pos = r.geopos('locations', 'Beijing')
print(f"北京坐标: {pos}")  # [(116.397, 39.908)]

北京坐标: [(116.39700025320053, 39.9079996716498)]


In [49]:
# 4. GEORADIUS: 以某坐标为中心，查找半径内的成员
nearby = r.georadius('locations', 116.0, 40.0, 200, unit='km', withdist=True)
print(f"附近200km内的城市: {nearby}")

附近200km内的城市: [[b'Beijing', 35.3618]]


In [50]:
# 5. GEORADIUSBYMEMBER: 以某个成员为中心，查找半径内的成员
nearby_beijing = r.georadiusbymember('locations', 'Beijing', 1000, unit='km', withdist=True)
print(f"北京1000km内的城市: {nearby_beijing}")

北京1000km内的城市: [[b'Beijing', 0.0]]


### Stream

In [53]:
# 1. XADD: 添加消息到流
msg_id = r.xadd('mystream', {'field1': 'value1', 'field2': 'value2'})
print(f"消息ID: {msg_id}")  # 输出: 1693847293123-0

消息ID: b'1788855047389-0'


In [54]:
# 2. XLEN: 获取流中消息数量
length = r.xlen('mystream')
print(f"流长度(消息数量): {length}")

流长度(消息数量): 3


In [55]:
# 3. XREAD: 读取消息（非阻塞）
messages = r.xread({'mystream': '0'}, count=10)  # 从ID=0开始读, 限制读取条数
# messages = r.xread({'mystream': '1788568162844-0'})  # 从ID=1788568162844开始读
messages

[[b'mystream',
  [(b'1788854981862-0', {b'field1': b'value1', b'field2': b'value2'}),
   (b'1788855045527-0', {b'field1': b'value1', b'field2': b'value2'}),
   (b'1788855047389-0', {b'field1': b'value1', b'field2': b'value2'})]]]

In [56]:
# 4. XRANGE: 按ID范围读取
messages = r.xrange('mystream', min='-', max='+', count=10)  # - 表示最小ID，+ 表示最大ID
messages

[(b'1788854981862-0', {b'field1': b'value1', b'field2': b'value2'}),
 (b'1788855045527-0', {b'field1': b'value1', b'field2': b'value2'}),
 (b'1788855047389-0', {b'field1': b'value1', b'field2': b'value2'})]

消息的消费组

In [59]:
stream_key = "mystream"
group_name = "mygroup"

In [61]:
# 删除消费组
result = r.xgroup_destroy('mystream', 'mygroup')
if result:
    print("消费组删除成功")
else:
    print("消费组不存在")

消费组删除成功


In [62]:
# 方式1：从流开头创建消费组
result = r.xgroup_create(
    name='mystream',           # 流名称
    groupname='mygroup',      # 消费组名称
    id='0'                     # 从最早的消息开始
)
result

True

In [ ]:
# 从消费组读取消息
# XREADGROUP GROUP mygroup consumer1 COUNT 10 BLOCK 5000
result = r.xreadgroup(
    groupname="mygroup",         # 消费组名
    consumername="customer",     # 消费者名称（唯一标识）
    streams={stream_key: '>'},   # '>' 表示只读取从未分发过的新消息
    count=10,                    # 每次最多读取10条
    block=5000                   # 阻塞5秒，无消息则超时
)
result

In [ ]:
# 消息确认
# 确认消息已处理（XACK）
if result:
    for stream_name, messages in result:
        for msg_id, data in messages:
            re = r.xack(stream_key, group_name, msg_id)  # 返回确认的消息数
            print(re, ":", F"customer消费者消息 {msg_id} 已经确认")

In [ ]:
import redis
client = redis.Redis(
    host='localhost', 
    port=6379,
    decode_responses=True  # 自动解码为字符串
)

# 获取待处理消息概览
pending_info = client.xpending("mystream", "mygroup")
pending_info

In [ ]:
result = client.xreadgroup(
    groupname="mygroup",         # 消费组名
    consumername="customer",     # 消费者名称（唯一标识）
    streams={stream_key: '>'},   # '>' 表示只读取从未分发过的新消息
    count=10,                    # 每次最多读取10条
    block=5000                   # 阻塞5秒，无消息则超时
)

In [ ]:
pending_info = client.xpending("mystream", "mygroup")
pending_info

订单于订单延时

In [63]:
import redis
import json
import time
import threading
from datetime import datetime, timedelta
from typing import Dict, Any, Optional

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

STREAM_KEY = 'order:stream'
GROUP_NAME = 'order_process_group'

# 创建消费者组（如果不存在）
try:
    r.xgroup_create(STREAM_KEY, GROUP_NAME, id='0', mkstream=True)
except redis.exceptions.ResponseError as e:
    if 'BUSYGROUP' not in str(e):
        raise
    print("消费者组已存在")

In [64]:
def create_order(order_id: str, user_id: str, product: str, amount: float):
    """
    创建订单，发送待支付消息到 Stream
    """
    # 1. 模拟保存订单到数据库
    order_data = {
        'order_id': order_id,
        'user_id': user_id,
        'product': product,
        'amount': amount,
        'status': 'pending',
        'created_at': datetime.now().isoformat(),
        'expire_after': 1800  # 30分钟超时（秒）
    }
    
    # 2. 发送消息到 Stream
    message_data = {
        'order_id': order_id,
        'user_id': user_id,
        'product': product,
        'amount': str(amount),
        'created_at': order_data['created_at'],
        'expire_after': '1800'
    }
    
    msg_id = r.xadd(STREAM_KEY, message_data)
    print(f"[订单服务] 订单 {order_id} 已创建，消息ID: {msg_id}")
    
    # 3. 模拟保存订单数据到 Redis Hash（便于后续查询）
    r.hset(f'order:detail:{order_id}', mapping=order_data)
    
    return order_data

# 测试：创建几个订单
create_order('ORD-1001', 'user_001', 'iPhone 15', 6999)
create_order('ORD-1002', 'user_002', 'MacBook Pro', 12999)
create_order('ORD-1003', 'user_003', 'AirPods Pro', 1899)

[订单服务] 订单 ORD-1001 已创建，消息ID: 1788856490829-0
[订单服务] 订单 ORD-1002 已创建，消息ID: 1788856490830-0
[订单服务] 订单 ORD-1003 已创建，消息ID: 1788856490832-0


{'order_id': 'ORD-1003',
 'user_id': 'user_003',
 'product': 'AirPods Pro',
 'amount': 1899,
 'status': 'pending',
 'created_at': '2026-09-08T16:34:50.831095',
 'expire_after': 1800}

In [ ]:
class OrderTimeoutProcessor:
    """订单超时处理器（消费者组模式）"""
    
    def __init__(self, consumer_name: str):
        self.consumer_name = consumer_name
        self.running = True
    
    def process_message(self, msg_id: str, msg_data: Dict[str, Any]) -> bool:
        """
        处理单条消息：检查订单是否已支付，未支付则取消
        返回 True 表示处理成功，可以 ACK
        """
        order_id = msg_data.get('order_id')
        if not order_id:
            return False
        
        print(f"[消费者 {self.consumer_name}] 处理订单 {order_id}")
        
        # 1. 查询订单状态
        order_detail = r.hgetall(f'order:detail:{order_id}')
        if not order_detail:
            print(f"  ⚠️ 订单 {order_id} 不存在")
            return True  # 订单已删除，视为已处理
        
        status = order_detail.get('status', 'pending')
        created_at = order_detail.get('created_at')
        
        # 2. 检查是否超时
        if created_at:
            created_time = datetime.fromisoformat(created_at)
            elapsed = (datetime.now() - created_time).total_seconds()
            expire_after = float(order_detail.get('expire_after', 1800))
            
            # 3. 业务逻辑：如果未支付且已超时，取消订单
            if status == 'pending' and elapsed > expire_after:
                # 更新订单状态为 "cancelled"
                r.hset(f'order:detail:{order_id}', 'status', 'cancelled')
                r.hset(f'order:detail:{order_id}', 'cancelled_at', datetime.now().isoformat())
                print(f"  ✅ 订单 {order_id} 已超时取消（耗时: {elapsed:.0f}s）")
                
                # 这里可以触发其他操作：
                # - 释放库存
                # - 发送通知
                # - 记录日志
            elif status == 'paid':
                print(f"  ℹ️ 订单 {order_id} 已支付，无需处理")
            else:
                print(f"  ℹ️ 订单 {order_id} 状态: {status}，未超时")
        
        return True
    
    def run(self):
        """消费者主循环：持续从 Stream 读取消息"""
        print(f"[消费者 {self.consumer_name}] 启动...")
        
        while self.running:
            try:
                # 从消费者组读取消息（阻塞 1 秒）
                results = r.xreadgroup(
                    GROUP_NAME,
                    self.consumer_name,
                    {STREAM_KEY: '>'},  # '>' 表示读取新消息
                    count=5,
                    block=1000  # 阻塞 1 秒
                )
                
                if not results:
                    continue
                
                # 处理每条消息
                for stream_name, messages in results:
                    for msg_id, msg_data in messages:
                        # 处理消息
                        success = self.process_message(msg_id, msg_data)
                        
                        # 处理成功后确认消息
                        if success:
                            r.xack(STREAM_KEY, GROUP_NAME, msg_id)
                            print(f"  ✅ 消息 {msg_id} 已确认 (ACK)")
                        else:
                            print(f"  ❌ 消息 {msg_id} 处理失败，将重新投递")
                            
            except Exception as e:
                print(f"错误: {e}")
                time.sleep(1)
    
    def stop(self):
        """停止消费者"""
        self.running = False
        print(f"[消费者 {self.consumer_name}] 已停止")

In [ ]:
# 创建并启动多个消费者（模拟多实例）
consumers = [
    OrderTimeoutProcessor('worker-1'),
    OrderTimeoutProcessor('worker-2'),
]

# 启动消费者线程
threads = []
for consumer in consumers:
    t = threading.Thread(target=consumer.run, daemon=True)
    t.start()
    threads.append(t)

print("\n消费者已启动，正在监听订单消息...")
print("提示：请运行模拟支付函数或等待订单超时\n")

# 模拟运行 35 秒（让订单超时）
time.sleep(35)

# 停止消费者
for consumer in consumers:
    consumer.stop()

# 查看最终订单状态
print("\n=== 最终订单状态 ===")
for order_id in ['ORD-1001', 'ORD-1002', 'ORD-1003']:
    detail = r.hgetall(f'order:detail:{order_id}')
    if detail:
        status = detail.get('status', 'unknown')
        print(f"{order_id}: {status}")

In [ ]:
def pay_order(order_id: str):
    """模拟用户支付订单"""
    detail = r.hgetall(f'order:detail:{order_id}')
    if not detail:
        print(f"订单 {order_id} 不存在")
        return False
    
    if detail.get('status') != 'pending':
        print(f"订单 {order_id} 当前状态: {detail.get('status')}，无法支付")
        return False
    
    # 更新订单状态为 paid
    r.hset(f'order:detail:{order_id}', 'status', 'paid')
    r.hset(f'order:detail:{order_id}', 'paid_at', datetime.now().isoformat())
    print(f"✅ 订单 {order_id} 已支付")
    return True

# 在程序运行过程中，可以调用此函数模拟支付
# 创建并启动多个消费者（模拟多实例）
consumers = [
    OrderTimeoutProcessor('worker-1'),
    OrderTimeoutProcessor('worker-2'),
]

# 启动消费者线程
threads = []
for consumer in consumers:
    t = threading.Thread(target=consumer.run, daemon=True)
    t.start()
    threads.append(t)

print("\n消费者已启动，正在监听订单消息...")
print("提示：请运行模拟支付函数或等待订单超时\n")

create_order('ORD-1001', 'user_001', 'iPhone 15', 6999)
create_order('ORD-1002', 'user_002', 'MacBook Pro', 12999)
create_order('ORD-1003', 'user_003', 'AirPods Pro', 1899)
time.sleep(3)
pay_order('ORD-1001')
pay_order('ORD-1002')
time.sleep(3)
print("\n=== 最终订单状态 ===")
for order_id in ['ORD-1001', 'ORD-1002', 'ORD-1003']:
    detail = r.hgetall(f'order:detail:{order_id}')
    if detail:
        status = detail.get('status', 'unknown')
        print(f"{order_id}: {status}")
for consumer in consumers:
    consumer.stop()

### JSON

RedisJSON 的核心是 `JSON.SET` 和 `JSON.GET`，配合 JSONPath 语法，可以对文档的任意部分进行精细操作。
支持XPATH语法，文档结构树搜索语法
| 方法 | 对应 Redis 命令 | 描述 |
| :--- | :--- | :--- |
| `json().set(key, path, obj)` | `JSON.SET` | 在指定路径设置 JSON 值。 |
| `json().get(key, *paths)` | `JSON.GET` | 获取指定路径的值。 |
| `json().delete(key, path)` | `JSON.DEL` | 删除指定路径的值。 |
| `json().type(key, path)` | `JSON.TYPE` | 获取指定路径值的类型。 |
| `json().arrappend(key, path, *args)` | `JSON.ARRAPPEND` | 向数组追加元素。 |
| `json().arrlen(key, path)` | `JSON.ARRLEN` | 获取数组长度。 |
| `json().arrpop(key, path, index=-1)` | `JSON.ARRPOP` | 移除并返回数组中的一个元素。 |
| `json().arrtrim(key, path, start, stop)` | `JSON.ARRTRIM` | 将数组修剪为指定范围。 |
| `json().numincrby(key, path, number)` | `JSON.NUMINCRBY` | 原子性地增加数字。 |
| `json().objkeys(key, path)` | `JSON.OBJKEYS` | 获取对象的所有键。 |
| `json().strappend(key, path, string)` | `JSON.STRAPPEND` | 向字符串追加内容。 |

XPATH简单应用
| 语法/符号 | 含义 | 示例 | 说明 |
| :--- | :--- | :--- | :--- |
| `$` | 根对象/元素 | `$` | 所有表达式都由此开始，指向整个 JSON 文档的根部。 |
| `@` | 当前对象/元素 | `@.price` | 在过滤器或脚本表达式中，代表正在被处理的元素。 |
| `.` 或 `[]` | 子元素操作符 | `$.store.book[0].title`<br>`$['store']['book'][0]['title']` | 两种写法等价，用于访问下级节点。如果键名包含特殊字符（如空格），则必须使用 `[]` 和引号，例如 `$['Log Level']`。 |
| `..` | 递归下降/深度扫描 | `$..author` | 不管对象在 JSON 的哪个层级，都会找出来。`$..author` 会匹配文档中所有名为 `author` 的字段。 |
| `*` | 通配符 | `$.store.book[*].author` | 匹配所有对象成员或数组元素，常用于获取数组里所有元素的某一属性。 |
| `[start:end:step]` | 数组切片 | `$..book[:2]` | 获取数组的子集。例如 `[:2]` 取前两个，`[-1:]` 取最后一个。 |
| `[?(expression)]` | 过滤器表达式 | `$..book[?(@.price < 10)]` | 使用布尔表达式筛选符合条件的数组元素，功能非常强大。 |

| 函数 | 描述 | 示例 |
| :--- | :--- | :--- |
| `min()` | 数值数组的最小值 | `$.store.book[*].price.min()` |
| `max()` | 数值数组的最大值 | `$.store.book[*].price.max()` |
| `avg()` | 数值数组的平均值 | `$.store.book[*].price.avg()` |
| `length()` | 数组的长度 | `$.store.book.length()` |

- JSON格式提供内部嵌套，且提供数据库级别的语法设计(XPATH)

In [14]:
# 准备一个 JSON 文档
user = {
    "id": 1001,
    "name": "Alice",
    "email": "alice@example.com",
    "address": {
        "city": "San Francisco",
        "country": "US"
    },
    "tags": ["python", "redis"],
    "score": 9.5
}

In [15]:
# 存储到 Redis，key 为 'user:1001'
r.json().set("user_info:1001", "$", user)

True

In [16]:
whole_doc = r.json().get("user_info:1001")
print(f"完整文档: {whole_doc}")

完整文档: {'id': 1001, 'name': 'Alice', 'email': 'alice@example.com', 'address': {'city': 'San Francisco', 'country': 'US'}, 'tags': ['python', 'redis'], 'score': 9.5}


In [17]:
# 使用 JSONPath 读取特定字段 (注意返回值是列表)
city = r.json().get("user_info:1001", "$.tags[0]")
print(f"城市: {city[0] if city else None}")

城市: python


In [ ]:
r.json().numincrby("user_info:1001", "$.score", 0.1)
print(f"再次更新后的分数: {r.json().get('user_info:1001', '$.score')}")

经典使用场景

In [18]:
# 用户 '123' 的购物车
cart_key = 'cart:123'

r.json().set(cart_key, '$', {
    'user_id': '123',
    'items': [],
    'total_price': 0,
    'updated_at': '2026-09-06T10:00:00Z'
})

True

In [19]:
# 添加第一个商品 (使用JSON.ARRAPPEND)
first_item = {'sku': 'A100', 'name': '机械键盘', 'price': 299.00, 'qty': 1}
r.json().arrappend(cart_key, '$.items', first_item)

[1]

In [20]:
# 更新总价
r.json().numincrby(cart_key, '$.total_price', 299.00)

[299.0]

In [21]:
# 添加第二个商品
second_item = {'sku': 'B200', 'name': '无线鼠标', 'price': 89.00, 'qty': 2}
r.json().arrappend(cart_key, '$.items', second_item)
r.json().numincrby(cart_key, '$.total_price', 178.00)

[477.0]

In [22]:
# 获取购物车中所有商品的名称
item_names = r.json().get(cart_key, '$.items[*].name')
print(item_names)  # 输出: ['机械键盘', '无线鼠标']

# 修改特定商品的数量 (将第一个商品数量改为2)
r.json().set(cart_key, '$.items[0].qty', 2)
# 重新计算总价 (此处为演示，实际需遍历计算)
r.json().set(cart_key, '$.total_price', 299*2 + 89*2)

# 移除第一个商品 (使用JSON.ARRPOP)
popped_item = r.json().arrpop(cart_key, '$.items', 0)
print(f"移除了商品: {popped_item}")  # 输出: 移除了商品: 机械键盘

# 获取最终的购物车
final_cart = r.json().get(cart_key)
print(final_cart)

['机械键盘', '无线鼠标']
移除了商品: ['{"sku":"A100","name":"机械键盘","price":299.0,"qty":2}']
{'user_id': '123', 'items': [{'sku': 'B200', 'name': '无线鼠标', 'price': 89.0, 'qty': 2}], 'total_price': 776, 'updated_at': '2026-09-06T10:00:00Z'}


### Time Series(时间序列)

- 内部自动生成时间

In [65]:
ts = r.ts()
ts

In [66]:
# 创建并写入数据 -------------
# 创建一个名为 'temperature:sensor:01' 的时间序列
# RETENTION 60000 表示数据保留 60 秒
# LABELS 为其添加标签，方便后续按 sensor_id 查询
ts.create(
    'temperature:sensor:01',
    retention_msecs=60000,
    labels={'sensor_id': '01', 'room': 'living_room'}
)

True

In [67]:
# 添加数据点：TS.ADD key timestamp value
# 使用 '*' 作为时间戳，会让 Redis 自动填入服务器当前时间
ts.add('temperature:sensor:01', '*', 23.5)
time.sleep(1)
ts.add('temperature:sensor:01', '*', 24.1)
time.sleep(1)
ts.add('temperature:sensor:01', '*', 23.8)

1788857226505

In [70]:
from datetime import datetime
dt = datetime.fromtimestamp(1788857226505/1000.0)
dt

datetime.datetime(2026, 9, 8, 16, 47, 6, 505000)

In [71]:
# 查询数据 ------------------
# 获取最新的一个数据点
last_value = ts.get('temperature:sensor:01')
print(f"最新数据: 时间戳={last_value[0]}, 值={last_value[1]}")

# 获取特定时间范围内的所有原始数据
# '-' 和 '+' 分别代表最小和最大时间戳
all_data = ts.range('temperature:sensor:01', '-', '+')
print("全部数据:", all_data)

# 获取特定时间范围，并按 5000 毫秒（5秒）进行平均值聚合
# 这会大大减少返回的数据量
aggregated_data = ts.range(
    'temperature:sensor:01',
    '-',
    '+',
    aggregation_type='avg',
    bucket_size_msec=5000
)
print("聚合数据 (5秒平均):", aggregated_data)

最新数据: 时间戳=1788857226505, 值=23.8
全部数据: [(1788857224501, 23.5), (1788857225503, 24.1), (1788857226505, 23.8)]
聚合数据 (5秒平均): [(1788857220000, 23.5), (1788857225000, 23.950000000000003)]


### Bloom/Cukoo(布隆/布谷鸟过滤器)

- 快速筛选，看是否存在
    - 用户登录
        - 所有用户名缓冲在Redis中。
            - 先走Redis

In [72]:
# 1. 布隆过滤器 (Bloom Filter) - 不支持删除
# 创建一个名为 'bloom_filter' 的过滤器，设定期望的误判率为 0.01 (1%)，预估元素数量为 10000
# 使用 bf().create() 方法 
r.bf().create('bloom_filter', 0.01, 10000)

True

In [ ]:
# 添加元素
r.bf().add('bloom_filter', 'user:12345')
# 批量添加元素 [citation:6]
r.bf().madd('bloom_filter', 'user:100', 'user:200')

In [ ]:
# 检查元素是否存在 (返回 1 表示可能存在，0 表示一定不存在)
is_exists = r.bf().exists('bloom_filter', 'user:12345')
print(f"布隆过滤器: user:12345 是否存在? {is_exists}")  # 输出: 1

In [ ]:
# 批量检查 
results = r.bf().mexists('bloom_filter', 'user:100', 'user:999')
print(f"布隆过滤器批量检查: {results}")  # 输出: [1, 0]

布谷

In [ ]:
r.cf().create('cuckoo_filter', 10000)

In [ ]:
r.cf().add('cuckoo_filter', '192.168.1.1')
# 布谷鸟过滤器使用 cf().insert() 进行批量添加 [citation:5][citation:6]
r.cf().insert('cuckoo_filter', ['192.168.1.2', '192.168.1.3'])

In [ ]:
# 检查元素是否存在
is_exists = r.cf().exists('cuckoo_filter', '192.168.1.1')
print(f"布谷鸟过滤器: 192.168.1.1 是否存在? {is_exists}")  # 输出: 1

In [ ]:
# 删除元素 (布隆过滤器不支持此操作) 
r.cf().delete('cuckoo_filter', '192.168.1.1')
is_exists_after_del = r.cf().exists('cuckoo_filter', '192.168.1.1')
print(f"布谷鸟过滤器删除后: 192.168.1.1 是否存在? {is_exists_after_del}")  # 大概率输出: 0

### Vector Set(向量集合)

```python
vs.vadd(
    key: 'KeyT',
    vector: 'list[float] | bytes',
    element: 'str',
    reduce_dim: 'int | None' = None,   
    # 设置要将向量缩减到的维度。如果未提供，则不缩减向量。
    cas: 'bool | None' = False,  
    # 指示在添加向量时是否使用 CAS（检查并设置风格）。如果未提供，则不使用 CAS。
    quantization: 'QuantizationOptions | None' = None,
    # 设置要使用的量化类型。如果未提供，则使用 int8 量化。 可选选项有：
    # NOQUANT：不量化
    # BIN：二进制量化
    # Q8：有符号 8 位量化
    ef: 'Number | None' = None,
    # 设置要使用的探索因子。如果未提供，则使用默认的探索因子。搜索算法相关
    attributes: 'dict | str | None' = None,
    # 是一个字典或 JSON 字符串，包含要为该向量设置的属性。如果未提供，则不设置任何属性。
    numlinks: 'int | None' = None,
    # 设置为该向量创建的链接数。如果未提供，则使用默认的链接数。
) -> 'Awaitable[int] | int'
```

```python
# 将向量或元素 input 与向量集合 key 中的其他向量进行比较。
vs.vsim(  
    key: 'KeyT',
    input: 'list[float] | bytes | str',
    with_scores: 'bool | None' = False,
    # 设置是否应在结果中返回每个元素的相似度分数。
    with_attribs: 'bool | None' = False,
    # 设置是否应在结果中返回元素的属性，若无属性则返回 None。
    count: 'int | None' = None,
    # 设置要返回的结果数量。
    ef: 'Number | None' = None,
    # ef 设置探索因子（search-exploration-factor）。性能调优参数
    # 它的作用是控制向量相似性搜索的“工作量”
    # 这个参数值越高，搜索时探索的节点越多，因此能更精确地找到最相似的结果（提高召回率）。
    #     |- 但相应的代价是，搜索的速度会变慢，性能会有所牺牲。
    #     |- 这是一个在搜索精度和性能之间做权衡的调优参数，常见的取值范围在50到1000之间，默认200
    filter: 'str | None' = None,
    # 设置应应用于搜索的过滤器。
    filter_ef: 'str | None' = None,
    # 设置最大过滤探努力值。引擎在尝试满足过滤条件上的"努力程度"或"尝试次数上限，
    # 如果你的过滤条件非常严格（比如 .rare_field == 1），导致结果很少，
    # 你可以通过提高 FILTER-EF 的值来迫使引擎扫描更多的候选向量以尝试找到满足条件的元素
    # FILTER-EF 的默认值与 COUNT 参数（你希望返回的结果数）相关。默认情况下，FILTER-EF 的值是 COUNT * 100
    truth: 'bool | None' = False,
    # 启用时，强制命令执行线性扫描。
    no_thread: 'bool | None' = False,
    # 启用时，强制命令在主线程中对数据结构执行搜索。
    epsilon: 'Number | None' = None,
    # 一个介于 0 和 1 之间的浮点数，如果指定，则仅返回距离不超过该指定值的元素。
) -> 'Awaitable[VSimResult] | VSimResult'
```

In [73]:
r.vset().vadd("points", [1.0, 1.0], "pt:A")
r.vset().vadd("points", [-1.0, -1.0], "pt:B")
r.vset().vadd("points", [-1.0, 1.0], "pt:C")

True

In [74]:
# 通过 execute_command（更通用） ： 2是向量维度
r.execute_command('VADD', 'points', 'VALUES', '2', '1.0', '-1.0', 'pt:D')

True

In [75]:
# 查询向量集信息 -------------------
# VCARD: 获取元素数量
count = r.vset().vcard("points")
# 或 count = r.execute_command('VCARD', 'points')
print(f"向量集元素数量: {count}")

向量集元素数量: 4


In [76]:
# 相似性搜索-------------------
# 查找与 'pt:A' 最相似的 2 个元素，返回相似度分数
similar = r.vset().vsim(
    "points", 
    input="pt:A", 
    with_scores=True, 
    count=2
)
print(f"与 pt:A 最相似的元素: {similar}")
# 输出格式: ['pt:E', 0.7071, 'pt:D', 0.0]
# 分数含义：余弦相似度，1 表示完全相同，0 表示正交

与 pt:A 最相似的元素: {'pt:A': 1.0, 'pt:D': 0.5}


In [77]:
# -----------更多的操作-----------
# VEMB: 获取元素的存储向量（可能经过量化）
vec = r.vset().vemb("points", "pt:A")
print(f"pt:A 的向量: {vec}")

# VREM: 删除元素
r.vset().vrem("points", "pt:C")

# VSETATTR / VGETATTR: 设置/获取元素的 JSON 元数据
r.vset().vsetattr("points", "pt:A", {"name": "Point A", "description": "First point"})
attrs = r.vset().vgetattr("points", "pt:A")
print(f"pt:A 的属性: {attrs}")

pt:A 的向量: [0.9999999403953552, 0.9999999403953552]
pt:A 的属性: {'name': 'Point A', 'description': 'First point'}


In [95]:
# 随机查询向量
result = r.vset().vrandmember("points")
result

'pt:D'

In [97]:
result = r.vset().vrange("points", start="-", end="+")
result

['pt:A', 'pt:B', 'pt:D']

In [98]:
for element_name in result:
    vector = r.vset().vemb("points", element_name)
    print(element_name, vector)

pt:A [0.9999999403953552, 0.9999999403953552]
pt:B [-0.9999999403953552, -0.9999999403953552]
pt:D [0.9999999403953552, -0.9999999403953552]


- 查询时，element_name这个元素名中文支持差，尽量用英文命名。

示例

In [99]:
import random

# 模拟标题及其向量（实际应用中需要调用嵌入模型生成）
news_data = {
    'news:1': {'title': '今日股市大涨，科技股领涨', 'vector': [random.random() for _ in range(4)]},
    'news:2': {'title': '最新iPhone发布，AI功能成亮点', 'vector': [random.random() for _ in range(4)]},
    'news:3': {'title': '全球气候变暖报告发布', 'vector': [random.random() for _ in range(4)]},
    'news:4': {'title': '足球世界杯中国队出线', 'vector': [random.random() for _ in range(4)]},
    'news:5': {'title': 'OpenAI发布新模型GPT-5', 'vector': [random.random() for _ in range(4)]},
}

vs_key = 'news_vectors'
for news_id, data in news_data.items():
    # 使用 execute_command 批量添加
    cmd = ['VADD', vs_key, 'VALUES', '4'] + [str(v) for v in data['vector']] + [news_id]
    r.execute_command(*cmd)
    
print(f"已加载 {len(news_data)} 条新闻向量")

已加载 5 条新闻向量


In [100]:
# 模拟用户查询向量（实际应来自模型输出）
# 这里用与 news:2 相近的向量来模拟查询“科技”类新闻
query_vector = news_data['news:2']['vector']

# 执行相似性搜索
search_result = r.vset().vsim(
    vs_key,
    input=query_vector,
    with_scores=True,
    count=3  # 返回最相似的 3 条
)
search_result
print("搜索结果 (最相似的3条新闻):")
for news_id, score in search_result.items():
    print(f"- {news_data[news_id]['title']} (相似度: {float(score):.4f})")

搜索结果 (最相似的3条新闻):
- 最新iPhone发布，AI功能成亮点 (相似度: 0.9976)
- 足球世界杯中国队出线 (相似度: 0.8941)
- 全球气候变暖报告发布 (相似度: 0.8753)


In [101]:
filtered_result = r.vset().vsim(  
    key=vs_key,
    input=query_vector,
    with_scores=True,
    with_attribs=True,
    count=3,
    ef=200,
    filter=".category == 'tech'",
    filter_ef="100",
)
filtered_result

{}